In [1]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
import json, os, time, re, sys
from random import uniform
import requests



# Cambio por url de pagina 1 para despues ir cambiando a cada pagina
BASE = "https://books.toscrape.com/"
url = urljoin(BASE, "catalogue/page-1.html")
next_page = url



rating_number = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
items = []


OPENLIB_SEARCH = "https://openlibrary.org/search.json"
GOOGLE_BOOKS = "https://www.googleapis.com/books/v1/volumes"

# funcion para obtener autores usando Open Library y Google Books
def get_authors(title, category, lang="en", timeout=2, max_authors=5):
    """
    Devuelve una LISTA de autores usando Open Library y como fallback Google Books.
    Siempre retorna al menos ["Desconocido"] si no encuentra nada.
    """
    def _norm(name: str) -> str:
        return re.sub(r"\s+", " ", name).strip()

    autores = []

    # 1) Open Library (puede traer múltiples)
    try:
        ol = requests.get(
            "https://openlibrary.org/search.json",
            params={"title": title, "limit": 3},
            timeout=timeout
        )
        ol.raise_for_status()
        docs = ol.json().get("docs", [])
        for d in docs:
            for a in d.get("author_name", []) or []:
                a = _norm(a)
                if a and a not in autores:
                    autores.append(a)
                    if len(autores) >= max_authors:
                        break
            if len(autores) >= max_authors:
                break
    except requests.RequestException as e:
        print(f"[⚠️] OpenLibrary error: {e}")

    # 2) Google Books (fallback, agrega si faltan)
    if len(autores) < max_authors:
        query = f'intitle:"{title}"'
        if category:
            query += f' subject:"{category}"'
        try:
            gb = requests.get(
                "https://www.googleapis.com/books/v1/volumes",
                params={
                    "q": query,
                    "maxResults": 3,
                    "orderBy": "relevance",
                    "langRestrict": lang
                },
                timeout=timeout
            )
            gb.raise_for_status()
            items = gb.json().get("items", []) or []
            for it in items:
                for a in it.get("volumeInfo", {}).get("authors", []) or []:
                    a = _norm(a)
                    if a and a not in autores:
                        autores.append(a)
                        if len(autores) >= max_authors:
                            break
                if len(autores) >= max_authors:
                    break
        except requests.RequestException as e:
            print(f"[⚠️] Google Books error: {e}")

    return autores if autores else ["Desconocido"]


# funcion para obtener el soup de una pagina o una url
def get_soup(url_pagina_soup):
    for intento in range(5):  # 🔹 ADICIÓN: reintentos
        try:
            resp = requests.get(url_pagina_soup, timeout=20)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")
        except requests.RequestException:
            print(f"[WARN] Falló {url_pagina_soup} (intento {intento+1}): {e}")
            time.sleep(1.2 * (intento + 1))
    raise RuntimeError(f"No pude obtener {url_pagina_soup}")



# mientras existan paginas siguientes, veridicado por boton next. otra forma de hacerlo es iterando
while next_page:

    soup = get_soup(next_page)
    books = soup.find_all("article", class_="product_pod")

    for b in books:


        # Title
        title = b.h3.a["title"]


        # Price
        # 💥💥💥 
        txtprice = b.find("p", class_="price_color").get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", txtprice))


        # URL
        href = b.h3.a["href"]
        book_url = urljoin(next_page, href)

        try:
            # detalle 
            soup_detail = get_soup(book_url)
        except RuntimeError as e:
            print(f"[SKIP] No pude abrir detalle: {book_url} -> {e}")
            continue   

        # category
        breadcrumb_link = soup_detail.select("ul.breadcrumb li a")
        category = breadcrumb_link[-1].get_text(strip=True) if len(breadcrumb_link) >= 3 else "Unknown"


        # Rating                   
        rating_tag = soup_detail.select_one("p.star-rating")
        classes = rating_tag.get("class", []) if rating_tag else []                        
        rating_word = next((c for c in classes if c in rating_number), "One")
        rating = rating_number.get(rating_word, 1)                         


        # Stock
        available = soup_detail.find("p", class_="instock availability").text.strip()
        if available == "In stock":
            in_stock = True
        else: 
            in_stock = False


        # description
        description_tag = soup_detail.find("div", id="product_description")
        description = (
            description_tag.find_next_sibling('p').text.strip()
            if description_tag else None
        )


        # 🔹 ADICIÓN (llamar API): buscar autores por título
        # Explicación:
        # - Usamos la función get_autor(title, category).
        # - Retorna una lista [] con los nombres de los autores o Desconocido.
        # - Se agrega como campo "author" en el item para persistirlo luego en JSON.
        authors = get_authors(title, category)
        # Pequeña pausa de cortesía para no saturar la API si hay muchos libros
        time.sleep(0.15)

        items.append({
            "title": title,
            "category": category,
            "rating": rating,
            "URL": book_url,
            "price": price,
            "stock": in_stock,
            "description": description,
            # 🔹 ADICIÓN (nuevo campo en el dataset): autores como lista
            # Explicación:
            # - Este campo nuevo te permitirá luego crear tablas 'autores' y 'libro_autor' (M:N) en tu DB.
            # - Mantenerlo como lista te conserva todos los coautores que reporte la API.
            "authors": authors
        })

    time.sleep(0.5)

    # para cada pagina del 1 al 5
    botton_next = soup.find("li", class_="next")
    if botton_next:
        href_next = botton_next.a["href"]
        next_page = urljoin(next_page, href_next)
        time.sleep(0.15)
    else:
        break


# Guardar en JSON
with open('libros_scrapeados.json', 'w', encoding='utf-8') as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# Leer desde JSON
with open('libros_scrapeados.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)
    print(f'Se guardaron {len(datos)} libros')
    print(datos[0])  # Mostrar el primero para validar estructura

# Abrir automáticamente el archivo (solo en Windows)
os.startfile('libros_scrapeados.json')


[⚠️] Google Books error: HTTPSConnectionPool(host='www.googleapis.com', port=443): Max retries exceeded with url: /books/v1/volumes?q=intitle%3A%22Sharp+Objects%22+subject%3A%22Mystery%22&maxResults=3&orderBy=relevance&langRestrict=en (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1020)')))
[⚠️] OpenLibrary error: HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=2)
[⚠️] OpenLibrary error: HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=2)
[⚠️] OpenLibrary error: HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=2)
[⚠️] OpenLibrary error: HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=2)
[⚠️] OpenLibrary error: HTTPSConnectionPool(host='openlibrary.org', port=443): Read timed out. (read timeout=2)
[⚠️] OpenLibrary error: HTTPSConnectionPool(host='openlibrary.org', port=443)

In [ ]:
import sqlite3

# 1. Conectamos con la base
# conn representa la conexión a la base de datos.
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()


# 2. Activamos las llaves foráneas (para relaciones entre tablas)
conn.execute("PRAGMA foreign_keys = ON;")   # PRAGMA es una directiva especial de SQLite para configurar opciones




# 3. Escribimos el DDL (definición de tablas)
DDL = """
CREATE TABLE IF NOT EXISTS categories (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    id          INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    url         TEXT NOT NULL UNIQUE,
    price       REAL NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    stock       INTEGER NOT NULL CHECK (stock IN (0,1)),
    description TEXT,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS book_author (
    book_id   INTEGER NOT NULL,
    author_id INTEGER NOT NULL,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
);
"""

# 4. Ejecutamos todas las sentencias
conn.executescript(DDL)

print("✅ Tablas creadas correctamente")

conn.close()


✅ Tablas creadas correctamente


In [2]:
import sqlite3
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()

#Ver que las tablas se crearon
# SELECT sirve para seleccionar datos de una base de datos
# FROM es para especificar la tabla de donde se obtienen los datos
# sqlite_master es una tabla interna que guarda la estructura de la base
# WHERE sirve para filtrar los resultados
# type='table' filtra solo las filas que representan tablas
cursor.execute('''SELECT name FROM sqlite_master WHERE type='table' ''')
# fetchall() obtiene todas las filas del resultado de la consulta
# fetchall() trae una tupla de todas las filas de la tabla o de la base de datos
tablas = cursor.fetchall()

print("Tablas en la base de datos: ")
print(tablas)

conn.close()

Tablas en la base de datos: 
[('categories',), ('authors',), ('books',), ('book_author',)]


### `2 formas de cargar datos de JSON a datos de python`

In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"              # cambia si tu DB se llama distinto
JSON_PATH = "libros_scrapeados.json"  # cambia si tu JSON se llama distinto

def cargar_categorias(db_path=DB_PATH, json_path=JSON_PATH):
    '''
    Existen varias formas de cargar un JSON en Python.
    Aquí usamos la librería estándar 'json' y 'pathlib' para manejar rutas.
    '''

    # 1. Cargar JSON
    # Path es una clase de pathlib para manejar rutas de archivos
    # Creamos un objeto path con la ruta del JSON
    caminito_json = Path(JSON_PATH)
    print(caminito_json.name)      # 'libros_scrapeados.json'
    print(caminito_json.suffix)    # '.json'
    print(caminito_json.parent)    # '.'   (significa el directorio actual)
    print(caminito_json.resolve(), '\n') # 'C:\Users\TuNombre\proyecto\libros_scrapeados.json'



    # read_text lee el contenido del archivo como texto
    # 💥💥💥 Nos sirve para cargar el contenido del JSON completo como una cadena de texto
    # nos devuelve una cadena str con todo el contenido del archivo 
        #   '[  {"title": "A Light in the Attic", "price": 51.77, "category": "Poetry"}, 
        #       {"title": "Tipping the Velvet", "price": 53.74, "category": "Fiction"}
        #    ]'
    # como un string gigante
    texto_json = caminito_json.read_text(encoding="utf-8")


    # la funcion json.loads convierte el texto plano (sting) en una estructura de datos de Python
    # en este caso, una lista de diccionarios
    data = json.loads(texto_json)  # 'data' será una lista de diccionarios (items)
    
    # comprobamos como se muestra
    print(data[0])
    print(data[1])
    print(data[0]["title"],'\n')




# Ejecutar solo este paso primero
cargar_categorias()


libros_scrapeados.json
.json
.
C:\Users\HP\Desktop\THE HUDDLE\challenge4\libros_scrapeados.json 

{'title': 'A Light in the Attic', 'category': 'Poetry', 'rating': 3, 'URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'price': 51.77, 'stock': False, 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read th

In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"              # cambia si tu DB se llama distinto
JSON_PATH = "libros_scrapeados.json"  # cambia si tu JSON se llama distinto

def cargar_categorias(db_path=DB_PATH, json_path=JSON_PATH):
    '''
    Existen varias formas de cargar un JSON en Python.
    Aquí usamos la opcion clasica con open() y json.load().
    '''

    # el with es un manejador de contexto que abre y cierra el archivo automáticamente
    # ejecuta el bloque de código dentro del with
        # open() abre el archivo en modo lectura ("r") con codificación utf-8   ("r" == read)
        # as f crea una variable f que representa el archivo abierto
        # con el with, al salir del bloque, el archivo se cierra automáticamente
    with open("libros_scrapeados.json", "r", encoding="utf-8") as f:
        texto_json = f.read()

    data = json.loads(texto_json)
    print(data[0])
    print(data[1])
    print(data[0]["title"],'\n')

    print(type(texto_json))  # <class 'str'>
    print(type(data))        # típicamente <class 'list'> o <class 'dict'>
    print(len(data))         # si es lista, te da cuántos ítems (libros) hay


# Ejecutar solo este paso primero
cargar_categorias()


{'title': 'A Light in the Attic', 'category': 'Poetry', 'rating': 3, 'URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'price': 51.77, 'stock': False, 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? Rockab

### Diferencias

#### path 
- con libreria path nos sirve para leer y escribir, siempre en str
- codigo mas corto y mas facil de manejo

#### with tradicional
- sin librerias, compatible con las versiones de python mas viejas
- Mayor control en el manejo de datos, se puede escribir-leer-binario-etc.
- ideal cuando necesitamos realizar varios procesos, abrir- escribir-manejar errores-leet-etc.

In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"              # cambia si tu DB se llama distinto
JSON_PATH = "libros_scrapeados.json"  # cambia si tu JSON se llama distinto

def cargar_categorias(db_path=DB_PATH, json_path=JSON_PATH):


    # 1. Cargar JSON
    data = json.loads(Path(JSON_PATH).read_text(encoding="utf-8"))
    print(f"Leídos {len(data)} libros desde el JSON ✅ \n")



    # 2. Conexión + FK on
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()     # cursor sirve para ejecutar comandos SQL y traer resultados




    '''
    3. Insertar categorías únicas

    # for item in data -> recorre la lista data, item guarda cada diccionario de cada libro
    # el primer item.get("category") -> obtiene el valor de la clave "category" del diccionario, si no existe devuelve None
    # el ultimo item.get("category") -> es la condicional que filta e incluye solo los que tienen categoría no nula
    # las llaves {} crean un conjunto (set) que automáticamente elimina duplicados

        Version simplificada sin filtro de nulos:
        categorias = { item["category"] for item in data }
    '''

    # 3. recopilar categorías del JSON
    categorias = { item.get("category") for item in data if item.get("category") }
    print(f"categorías : {categorias}")


    '''
    4. Insertar en la tabla categories
    executemany ejecuta varias veces una sentencia SQL con diferentes parámetros
    Diferentes parametros se pasan como una lista de tuplas, cada tupla es un conjunto de valores para una ejecución

        version manual con bucle for:
        for categoria in categorias:
            cur.execute(
                "INSERT OR IGNORE INTO categories(name) VALUES (?);", (categoria,))
    
    
    INSERT OR IGNORE INTO categories(name) VALUES (?);
    - INSERT : inserta una nueva fila
    - OR IGNORE : si ya existe (violación de UNIQUE), ignora el error y no inserta
    - INTO categories(name) : especifica la tabla y columna donde insertar
    - VALUES (?) : el signo ? es un marcador de posición para el valor a insertar
        - (?) es un marcador de posición, un espacio reservado donde despues se va a insertar un valor de forma segura, sin mezclar texto python con SQL directamente
    



    [(c,) for c in sorted(categorias)]
        for c in sorted(categorias) : recorre cada categoría ordenada alfabéticamente
            sorted() devuelve una lista ordenada

        (c,) : crea una tupla con un solo elemento c (la categoría)
        [ ... ] : crea una lista de todas esas tuplas

        ejemplo de lo que genera
        [("Ficción",), ("Infantil",), ("Viajes",)]

    '''

    # 4. Insertar en la tabla categories
    cur.executemany( "INSERT OR IGNORE INTO categories(name) VALUES (?);", [(c,) for c in sorted(categorias)])




    # 5. Confirmamos los cambios en la conexion
        # sirve por si algo falla en el camino, se puede deshacer con conn.rollback() si no se confirmo
        # aumenta rendimiento, si se realiza de una sola vez, agrupar muchas operaciones y confirmar "en bloque" es mucho mas rapido que grabar una por una
    
    # Moral de la historia: “insertaste → confirmá”. Y si algo salió mal: “rollback y a casa”.
    conn.commit()
    



    # 6. Comprobar el resultado
    '''
        cur.execute("SELECT COUNT(*) FROM categories;")

            cur.execute ejecuta la consulta
                SELECT == seleciona. selecciona que devolver
                COUNT(*) == cuenta cuantas filas hay
                FROM == desde la tabla
                ; == fin de la sentencia

                fetchone()[0] == recupera la siguiente fila del iterador

            fetchone() trae la siguiente fila del resultado 
            [0] toma el primer elemento de la tupla, te devuelve un entero

    '''


        # fetchone o cualquier fetch devuelve siempre una secuencia en tupla o lista. nunca un valor suelto
    total = cur.execute("SELECT COUNT(*) FROM categories;").fetchone()[0]
    print(f"Listo categorías. Total en tabla: {total}")
    conn.close()

# Ejecutar solo este paso primero
cargar_categorias()


Leídos 1000 libros desde el JSON ✅ 

Listo categorías. Total en tabla: 50
